In [1]:
import os
import pandas as pd
import numpy as np
import dblp
from crossref.restful import Works
import requests
from serpapi import GoogleSearch

In [2]:
crossref_work = Works()

In [3]:
raw_dfs = [
    "data/cleaned/edu/search-results-master-v2.csv",
]

In [4]:
# Prepare DataFrame to store the results
columns = [
    "PaperTitle",
    "DOI",
    "Authors",
    "Abstract",
    "Publisher",
    "SemanticScholarUrl",
    "DoiUrl",
    "PublicationDate",
    "FieldOfStudy",
    "Conference-Journal",
    "PublicationTypes",
    "SearchString",
    "CitationCount",
    "SearchedFrom",
]

In [5]:
sch_fields = [
    "title",
    "externalIds",
    "authors",
    "abstract",
    "url",
    "publicationDate",
    "fieldsOfStudy",
    "venue",
    "publicationTypes",
    "citationCount",
    "externalIds",
]

In [6]:
# Helpers
def extract_sch(api_url, sch_fields):
    headers = {"Content-Type": "application/json"}
    params = {
        "fields": ",".join(sch_fields),
    }
    try:
        response = requests.get(api_url, headers=headers, params=params)
        result = response.json()
        return result
    except Exception as e:
        print(e)
        return {"error": e}


def get_affiliations_google_scholar(author_name):
    params = {
        "engine": "google_scholar_profiles",
        "mauthors": author_name.strip(),
        "api_key": os.environ.get("GOOGLE_SCHOLAR_API_KEY"),
    }

    try:
        search = GoogleSearch(params)
        results = search.get_dict()
        if (
            results.get("search_metadata", {}).get("status") == "Error"
            or len(results.get("profiles", [])) == 0
        ):
            return ["No Affiliation"]
        else:
            return results["profiles"][0]["affiliations"]
    except Exception as e:
        print(e)
        return ["No Affiliation"]


def extract_authors(sch_paper, crossref_paper):
    authors = None
    if crossref_paper is not None:
        authors = crossref_paper.get("author")
    if authors is not None:
        for i in range(len(authors)):
            author = authors[i]
            author_name = author.get("given", "") + " " + author.get("family", "")
            affiliations = author.get("affiliation", [])
            school_names = (
                [affil.get("name") for affil in affiliations]
                if affiliations
                else get_affiliations_google_scholar(author_name)
            )
            # Create a new dictionary with only 'name' and 'affiliation'
            authors[i] = {
                "name": author_name.strip(),
                "affiliation": school_names,
            }
    else:
        authors = sch_paper.get("authors")
        for i in range(len(authors)):
            author = authors[i]
            affiliations = author.get("affiliations")
            if affiliations is None:
                author_name = author["name"]
                affiliations = get_affiliations_google_scholar(author_name)

            authors[i] = {
                "name": author_name.strip(),
                "affiliation": affiliations,
            }

    return authors

In [9]:
def extract_data(row):
    pid = row["ID"]
    if "URL" in pid and "abs-" in pid:
        pid = "arXiv:" + pid.split("abs-")[1].replace("-", ".")

    api_url = f"https://api.semanticscholar.org/graph/v1/paper/{pid}"

    sch_paper = extract_sch(api_url, sch_fields)

    if sch_paper.get("error") is not None:
        # if can't search with ID, search by paper title

        api_url = f"https://api.semanticscholar.org/graph/v1/paper/search?query={row['PaperTitle']}"

        sch_paper = extract_sch(api_url, sch_fields)

        if len(sch_paper.get("data", [])) == 0:
            return {
                "PaperTitle": row["PaperTitle"],
                "DOI": row["ID"],
                "Authors": None,
                "Abstract": None,
                "Publisher": None,
                "SemanticScholarUrl": None,
                "DoiUrl": None,
                "PublicationDate": None,
                "FieldOfStudy": None,
                "Conference-Journal": None,
                "PublicationTypes": None,
                "SearchString": row["SearchString"],
                "CitationCount": None,
                "SearchedFrom": row["SearchedFrom"],
            }
        else:
            sch_paper = sch_paper["data"][0]

    doi = sch_paper.get("externalIds", {}).get("DOI", None)

    if doi is None and str(row["ID"]).startswith("DOI"):
        doi = row["ID"].split(":")[1]

    try:
        crossref_paper = crossref_work.doi(doi)

    except Exception as e:
        crossref_paper = None

    title = row["PaperTitle"]

    authors = extract_authors(sch_paper, crossref_paper)

    abstract = sch_paper.get("abstract", None)

    sch_url = sch_paper.get("url", None)

    doi_url = f"https://doi.org/{doi}"

    publication_date = sch_paper.get("publicationDate", None)

    fields_of_study = sch_paper.get("fieldsOfStudy", [])

    venue = sch_paper.get("venue", None)

    # publisher

    if crossref_paper is not None:
        publisher = crossref_paper.get("publisher")

    elif doi and "arxiv" in doi.lower():
        publisher = "arXiv"
    else:
        publisher = None

    # paper type

    if crossref_paper is not None:
        paper_type = [crossref_paper.get("type")]
    else:
        paper_type = sch_paper.get("publicationTypes", [])

    citation_count = sch_paper.get("citationCount", None)

    # TODO: paper keywords missing

    # TODO: paper type is conference/journal for arxiv papers

    # TODO: conference-journal name mismatch with publisher, i.e., for paper with name"ChatGPT in education: A discourse analysis of worries and concerns on social media", the conference name is "International Conference on Artificial Intelligence in Education", but the publisher is "Arxiv" (becauseit queryed from arxiv), need "Springer" instead.

    new_paper = {
        "PaperTitle": title,
        "DOI": doi,
        "Authors": authors,
        "Abstract": abstract,
        "Publisher": publisher,
        "SemanticScholarUrl": sch_url,
        "DoiUrl": doi_url,
        "PublicationDate": publication_date,
        "FieldOfStudy": fields_of_study,
        "Conference-Journal": venue,
        "PublicationTypes": paper_type,
        "SearchString": row["SearchString"],
        "CitationCount": citation_count,
        "SearchedFrom": row["SearchedFrom"],
    }

    return new_paper

In [10]:
for raw_df_path in raw_dfs:
    raw_df = pd.read_csv(raw_df_path)
    total_rows = len(raw_df)
    results = []
    
    # raw_df = raw_df.iloc[2452:]

    for index, row in raw_df.iterrows():
        # if index >= 5:
        #     break
        print(f"Processing {raw_df_path}: row {index + 1}/{total_rows}...")
        paper = extract_data(row)
        results.append(paper)

    # Creating a DataFrame from the results
    results_df = pd.DataFrame(results, columns=columns)

    # Generating a new file name based on the raw data file name
    new_file_name = raw_df_path.replace(".csv", "-full.csv")
    # new_file_name = 'test.csv'

    # Saving to a CSV file
    results_df.to_csv(new_file_name, index=False, mode="a", header=False)

Processing data/cleaned/edu/search-results-master-v2.csv: row 2453/2944...
Processing data/cleaned/edu/search-results-master-v2.csv: row 2454/2944...
Processing data/cleaned/edu/search-results-master-v2.csv: row 2455/2944...
Processing data/cleaned/edu/search-results-master-v2.csv: row 2456/2944...
Processing data/cleaned/edu/search-results-master-v2.csv: row 2457/2944...
Processing data/cleaned/edu/search-results-master-v2.csv: row 2458/2944...
Processing data/cleaned/edu/search-results-master-v2.csv: row 2459/2944...
Processing data/cleaned/edu/search-results-master-v2.csv: row 2460/2944...
Processing data/cleaned/edu/search-results-master-v2.csv: row 2461/2944...
Processing data/cleaned/edu/search-results-master-v2.csv: row 2462/2944...
Processing data/cleaned/edu/search-results-master-v2.csv: row 2463/2944...
Processing data/cleaned/edu/search-results-master-v2.csv: row 2464/2944...
Processing data/cleaned/edu/search-results-master-v2.csv: row 2465/2944...
Processing data/cleaned/e

# Testing stuff

In [37]:
len(results)

179

In [14]:
api_url = "https://api.semanticscholar.org/graph/v1/paper/search/"
headers = {
    "Content-Type": "application/json",
    "x-api-key": "X48LIBLqr86ouHlnMYd3z052sgEm3Nd2wMORPzu5",
}
fields = ["title", "externalIds", "paperId", "url"]
sch_search_string = "Saving the Vicuna: The Political, Biophysical, and Cultural History of Wild Animal Conservation in Peru, 1964-2000"
params = {"query": sch_search_string, "year": "2019-", "fields": ",".join(fields)}
response = requests.get(api_url, headers=headers, params=params)
result = response.json()
print(result)

{'total': 1, 'offset': 0, 'data': [{'paperId': '64fa0be92833e344e79f205976fb77a489461a58', 'externalIds': {'MAG': '3005763097', 'DOI': '10.1093/ahr/rhz939', 'CorpusId': 213377959}, 'url': 'https://www.semanticscholar.org/paper/64fa0be92833e344e79f205976fb77a489461a58', 'title': 'Saving the Vicuña: The Political, Biophysical, and Cultural History of Wild Animal Conservation in Peru, 1964–2000'}]}


In [12]:
pid = "ArXiv:2205.07144"
api_url = f"https://api.semanticscholar.org/graph/v1/paper/{pid}"
sch_paper = extract_sch(api_url, sch_fields)
sch_paper

{'paperId': 'cb179c916e73c7a6b0edf973b4ed087fe6224722',
 'externalIds': {'DBLP': 'conf/nips/LiBY22',
  'ArXiv': '2205.07144',
  'CorpusId': 248811155},
 'url': 'https://www.semanticscholar.org/paper/cb179c916e73c7a6b0edf973b4ed087fe6224722',
 'title': 'Network change point localisation under local differential privacy',
 'abstract': 'Network data are ubiquitous in our daily life, containing rich but often sensitive information. In this paper, we expand the current static analysis of privatised networks to a dynamic framework by considering a sequence of networks with potential change points. We investigate the fundamental limits in consistently localising change points under both node and edge privacy constraints, demonstrating interesting phase transition in terms of the signal-to-noise ratio condition, accompanied by polynomial-time algorithms. The private signal-to-noise ratio conditions quantify the costs of the privacy for change point localisation problems and exhibit a different

In [ ]:
# merge the three new csv, on paper title and doi